# Nhập bộ keyframe + objects của BTC lên Modal Volume `keyframes-btc`

Kế hoạch: [`docs/triet/ke-hoach-offline-ingest.md`](../../docs/triet/ke-hoach-offline-ingest.md) §7, §7.1

**Bấm Run All. Bấm lại nếu phiên đứt** — video nào xong sẽ bị bỏ qua.

**Đây là lượt 1 của 3** trong thư mục `notebooks/btc/`: `02` nhập → `03` SigLIP+OCR → `04`
subtitle+CLAP. Cả ba cùng volume, cùng bundle, cùng cách chia người.

### Đây là bundle ĐỐI CHỨNG, không phải bundle chính

Volume **riêng** `keyframes-btc` (v2). Không dùng chung với bundle DAKE: ảnh và `objects/`
đều đánh địa chỉ theo số thứ tự `n`, nên trộn hai bộ là bundle không còn khớp
`map-keyframes` của chính nó (§1.2).

Làm bây giờ vì tham số DAKE còn phải hiệu chỉnh tiếp, còn bộ này **không phụ thuộc ρ hay
ngưỡng mật độ** — nó là chỗ dựa an toàn và là mốc so sánh cho §3.7. Nhưng nó **không phải bản
nháp**: keyframe của nó cố định, nên mọi thứ trích trên nó là kết quả cuối.

### Không GPU, không giải mã — chuyển dữ liệu thuần

| gói | dung lượng | nội dung |
|---|---|---|
| `Keyframes_*.zip` ×14 | **28,69 GB** | 177.321 ảnh, STORED, `keyframes/{video}/{n:03d}.jpg` |
| `objects-aic25-b1.zip` | 0,60 GB | 177.321 json, DEFLATE → 1,68 GB |
| `map-keyframes-aic25-b1.zip` | ~1 MB | 873 csv |
| `media-info-aic25-b1.zip` | ~1 MB | 873 json |

Số file `objects` khớp **chính xác** số keyframe — hai thứ dùng chung một `n`, và tầng nhập
từ chối nếu hai con số lệch nhau.

| người | GB | video | zip |
|---|---|---|---|
| triet | 5,67 | 88 | L25 |
| trong | 6,02 | 122 | L22, L23, L24, L29 |
| quang | 5,67 | 153 | L21, L26_c, L28 |
| bao | 5,76 | 295 | L26_a, L26_d, L30 |
| dat | 5,57 | 215 | L26_b, L26_e, L27 |

### Nếu tải chậm

Host `aic-data.ledo.io.vn` có bóp băng thông, và 5 người cùng kéo thì tranh nhau cùng một
đường. `wget` ở đây có `--read-timeout=60` nên kết nối chết sẽ tự nối lại, nhưng nếu tốc độ
**đều** ở mức vài KB/s thì đó là bị bóp, không phải đứt — restart session để đổi IP Kaggle,
và hẹn nhau chạy lệch giờ.

`-c` + kiểm kích thước nghĩa là chậm chỉ tốn thời gian, không hỏng dữ liệu.

### Cần có trước khi chạy

| | |
|---|---|
| Kaggle Secrets | `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`, `GITHUB_TOKEN` |
| Settings | Internet **On**. Accelerator: **bất kỳ** — tầng này không chạm GPU |

## Cập nhật phân công và ZIP cache (19-08-2026)

Bảng 5 người ở phần mô tả phía trên là thông tin cũ. Cấu hình hiện hành chỉ chia cho `trong`
và `quang`; `trong` có 404 video thuộc L21, L23, L24, L25, L26_b, L29 và L30.

Notebook ưu tiên ZIP nguyên bản từ mọi private Kaggle Dataset đã attach trong
`/kaggle/input`. Nó kiểm tra tên, kích thước (khi inventory có khai báo) và central
directory của ZIP; nếu thiếu cache mới tải từ website. ZIP cache là read-only nên không bị
copy sang `/kaggle/working` hoặc xóa sau khi chạy. Xem `kaggle-zip-cache.md`.


## Step 0a — repo + gói

In [ ]:
MEMBER      = "missing1"          # <-- ĐỔI THÀNH TÊN BẠN
SHARDS      = "config/ingest_shards_batch2_missing.yaml"
TRANSPORT   = "modal"
VOLUME      = "keyframes-batch-2-v2"  # volume RIÊNG, không phải aic2026
VOL_VERSION = 2
REPO_URL    = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF    = "feature/triet/structure"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not REPO.is_dir():
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                       check=True, capture_output=True)   # token không lọt vào log
sys.path.insert(0, str(REPO / "src"))

# `nomodel` chứ không phải `keyframes`: tầng nhập không nạp model nào, và cùng
# file này được cả notebook DAKE dùng — nên tên nó không thuộc phương án nào.
# Các file requirements nằm ở gốc `notebooks/`, không nằm trong btc/ hay dake/.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO / "notebooks" / "requirements-kaggle-nomodel.txt")], check=True)
print("cài xong")

## Step 0b — Modal, phân công, việc đã xong

In [ ]:
import logging, yaml
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout, BundleWriter, StageStatus, load_shard_plan
from mfe.offline.stages import STAGES, StageContext
from mfe.offline.transport import build_transport

secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

plan      = load_shard_plan(REPO / SHARDS)
layout    = BundleLayout(plan.bundle)
transport = build_transport(TRANSPORT, volume=VOLUME,
                            local_root="/kaggle/working/bundle_local",
                            version=VOL_VERSION)
WORK   = Path("/kaggle/working/scratch"); WORK.mkdir(parents=True, exist_ok=True)
ZIPS   = Path("/kaggle/working/zips");    ZIPS.mkdir(parents=True, exist_ok=True)
KAGGLE_INPUT = Path("/kaggle/input")       # private Kaggle Datasets, read-only
writer = BundleWriter(transport, layout, MEMBER, WORK)

assert plan.keyframe_set == "btc"

EXTRAS    = yaml.safe_load((REPO / SHARDS).read_text(encoding="utf-8"))["extras"]

MY_VIDEOS = plan.videos_for(MEMBER)
status    = StageStatus.load(transport, layout, MEMBER)
LOCAL_MAPPING = {
    "N025-V002": Path("/kaggle/input/datasets/nhatlinh3005/keyframes/Keyframes_N021-N030/keyframes/N025-V002"),
    "N041-V001": Path("/kaggle/input/datasets/nhatlinh3005/keyframes/Keyframes_N041-N050/keyframes/N041-V001"),
    "S01-V007": Path("/kaggle/input/datasets/nhatlinh3005/keyframes/Keyframes_S01/keyframes/S01-V007")
}
MAP_DIR = Path("/kaggle/input/datasets/nhatlinh3005/map-keyframes-b2/map-keyframes")
todo      = [v for v in MY_VIDEOS if not status.is_done("n0_import_btc", v)]

print(f"bundle  : {plan.bundle}  ->  volume {VOLUME} (v{VOL_VERSION})")
print(f"người   : {MEMBER}")
print(f"video   : {len(MY_VIDEOS)} (tổng {len(plan.videos)})")
print(f"còn lại : {len(todo)}")

In [ ]:
# Các hàm download ZIP đã được loại bỏ do sử dụng Kaggle dataset trực tiếp


## Debug S01-V007 path
Chạy ô dưới đây nếu bạn gặp lỗi thiếu file keyframe cho S01-V007 để tìm đường dẫn đúng.

In [ ]:
import glob
print("=== CHECK PATH ===")
s01_base = "/kaggle/input/datasets/nhatlinh3005/keyframes/Keyframes_S01"
print("Bên trong Keyframes_S01 có gì:")
for p in glob.glob(f"{s01_base}/*"):
    print("  ", p)
print("\nBên trong Keyframes_S01/keyframes có gì:")
for p in glob.glob(f"{s01_base}/keyframes/*"):
    print("  ", p)
print("\nTìm file ảnh của S01-V007 (in 5 file đầu):")
images = glob.glob(f"{s01_base}/**/*.jpg", recursive=True)
if images:
    print("TÌM THẤY ẢNH Ở:", '/'.join(images[0].split('/')[:-1]))
    for img in images[:5]:
        print("  ", img)
else:
    print("KHÔNG TÌM THẤY BẤT KỲ FILE .jpg NÀO!")

## Step 1 — vòng lặp

Mỗi gói keyframe: tải (có resume) → với từng video: bung ảnh + objects + `map-keyframes` →
đẩy lên Volume → **rồi mới cắm mốc** → xoá sạch. Xong gói thì xoá file zip.

Tầng nhập từ chối nếu thiếu ảnh hoặc nếu số file `objects` khác số ảnh — hai thứ đó đánh địa
chỉ theo cùng một `n`, và lệch nhau là bundle tự mâu thuẫn.

In [ ]:
import time, traceback

failures = []
started = time.perf_counter()
done_count = 0

pending = [v for v in MY_VIDEOS if not status.is_done("n0_import_btc", v)]

if not pending:
    print()
    print("=== Đã xong hết, không còn video nào cần xử lý ===")
else:
    print()
    print(f"=== Còn {len(pending)} video cần đẩy lên ===")

for video_id in pending:
    try:
        kf_dir = LOCAL_MAPPING[video_id]
        map_csv = MAP_DIR / f"{video_id}.csv"
        
        ctx = StageContext(video_id, Path("/dev/null"), [], writer, layout, WORK,
                           params={"keyframes_dir": str(kf_dir),
                                   "map_csv": str(map_csv)})
        marker = STAGES["n0_import_btc"](ctx)
        status.done.add(("n0_import_btc", video_id))
        writer.write_part_manifest(video_id, [marker], keyframes=marker["keyframes"], merge=True)
        done_count += 1
        rate = (time.perf_counter() - started) / done_count
        left = (len(todo) - done_count) * rate / 3600
        print(f"[{done_count:>3}/{len(todo)}] {video_id}  {marker['keyframes']:>4} kf  "
              f"{marker['bytes'] / 2**20:>6.0f} MB  "
              f"{marker['elapsed_s']}s | còn ~{left:.1f}h")
    except Exception as err:
        failures.append((video_id, repr(err)))
        print(f"[LỖI] {video_id}: {err}")
        traceback.print_exc(limit=3)

print()
print("=" * 72)
print(f"phiên này: {done_count} video xong, {len(failures)} lỗi, "
      f"{(time.perf_counter() - started) / 3600:.1f}h")
for video_id, err in failures:
    print(f"  {video_id}: {err}")


## Step 2 — chốt phiên

In [ ]:
fresh = StageStatus.load(transport, layout, MEMBER)
left  = [v for v in MY_VIDEOS if not fresh.is_done("n0_import_btc", v)]

print(f"{MEMBER}: {len(MY_VIDEOS) - len(left)}/{len(MY_VIDEOS)} video đã lên {VOLUME}")
if left:
    print(f"còn {len(left)} — bấm Run All lại:")
    print("  " + ", ".join(left[:12]) + (" …" if len(left) > 12 else ""))
else:
    print("XONG phần của bạn. Báo Lead.")
    print(f"  python scripts/verify_bundle.py --shards {SHARDS} --member {MEMBER}")